In [17]:
import numpy as np
from scipy.special import erfcx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from numba import njit
import math
from tqdm import tqdm
import glob
import os
import pandas as pd

In [18]:
PI = np.pi
#parametros sistema
v = 0.0
D0 = 1.0
Dq = 2.0*D0
#parametros simulacion
n_part = 1000

#pasos temporales
dtt = 0.00025  #termalizacion
dtp = 0.00025   #toma de datos

#contadores de la simulacion
nct = 100000
ncp = 500000
ncet = 10
ncep = 10
ncgr = 100
ncsq = 500
ncxyz = 500 
n_cuad = ncp // ncep  

#cascara radial
dr = 0.005 #grosor cascara

#arreglos
x = np.zeros(n_part)
y = np.zeros(n_part)
z = np.zeros(n_part)
fx = np.zeros(n_part)
fy = np.zeros(n_part)
fz = np.zeros(n_part)
cfgx = np.zeros((n_cuad, n_part))
cfgy = np.zeros((n_cuad, n_part))
cfgz = np.zeros((n_cuad, n_part))
print(f"cfgx/y/z: {n_cuad} x {n_part} -> {3*cfgx.nbytes/1e9:.2f} GB en RAM")

q_max = 25
t = np.zeros(n_cuad)


#termodinamica

eex = 0.0
sige = 0.0
nprom = 0 #contador de config independ. promediadas

direcciones_base = np.array([
    # Ejes principales (Caras del cubo)
    [1.0, 0.0, 0.0], [-1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0], [0.0, -1.0, 0.0],
    [0.0, 0.0, 1.0], [0.0, 0.0, -1.0],
    
    # Diagonales planas (Bordes del cubo), normalizados a 1
    [1.0, 1.0, 0.0]/np.sqrt(2.0), [-1.0, -1.0, 0.0]/np.sqrt(2.0),
    [1.0, 0.0, 1.0]/np.sqrt(2.0), [-1.0, 0.0, -1.0]/np.sqrt(2.0),
    [0.0, 1.0, 1.0]/np.sqrt(2.0), [0.0, -1.0, -1.0]/np.sqrt(2.0),
    [-1.0, 1.0, 0.0]/np.sqrt(2.0), [1.0, -1.0, 0.0]/np.sqrt(2.0),
    [-1.0, 0.0, 1.0]/np.sqrt(2.0), [1.0, 0.0, -1.0]/np.sqrt(2.0),
    [0.0, -1.0, 1.0]/np.sqrt(2.0), [0.0, 1.0, -1.0]/np.sqrt(2.0),
    
    # Diagonales espaciales (Esquinas del cubo), normalizados a 1
    [1.0, 1.0, 1.0]/np.sqrt(3.0), [-1.0, -1.0, -1.0]/np.sqrt(3.0),
    [1.0, -1.0, 1.0]/np.sqrt(3.0), [-1.0, 1.0, -1.0]/np.sqrt(3.0),
    [-1.0, 1.0, 1.0]/np.sqrt(3.0), [1.0, -1.0, -1.0]/np.sqrt(3.0),
    [-1.0, -1.0, 1.0]/np.sqrt(3.0), [1.0, 1.0, -1.0]/np.sqrt(3.0)
    
])

cfgx/y/z: 50000 x 1000 -> 1.20 GB en RAM


In [19]:
@njit
def erfcx_numba(x):
    """
    Aproximación numérica de erfcx(x) = exp(x^2) * erfc(x) segura para Numba.
    Evita el error de desbordamiento de memoria (overflow) para x > 26.6.
    """
    if x < 25.0:
        return math.exp(x**2) * math.erfc(x)
    else:
        term1 = 1.0 / x
        term2 = 1.0 / (2.0 * x**3)
        term3 = 3.0 / (4.0 * x**5)
        return (1.0 / math.sqrt(math.pi)) * (term1 - term2 + term3)

In [20]:
@njit
def cdf_F_numba(q, q0, dt, v, Dq):
    den_raiz = math.sqrt(4.0 * Dq * dt)
    C = (q0 + v * dt) / den_raiz
    B = (q + q0 + v * dt) / den_raiz
    A = (v * q) / Dq
    razon = math.exp(A + C*C - B*B) * erfcx_numba(B) / erfcx_numba(C)
    return 1.0 - razon

In [ ]:
@njit
def brent_numba(a, b, q0, dt, v, Dq, x_rand, tol=1e-12, max_iter=100):
    fa = cdf_F_numba(a, q0, dt, v, Dq) - x_rand
    fb = cdf_F_numba(b, q0, dt, v, Dq) - x_rand

    if fa == 0.0:
        return a
    if fb == 0.0:
        return b
    if fa * fb >= 0.0:
        raise ValueError("La funcion debe tener signos opuestos en los extremos.")

    if abs(fa) < abs(fb):
        a, b = b, a
        fa, fb = fb, fa

    c = a
    fc = fa
    d = 0.0
    s = b          # inicializada aqui: nunca queda sin definir
    mflag = True
    i = 0

    while i < max_iter and abs(b - a) > tol:
        if fa != fb and fa != fc and fb != fc:
            s = (a * fb * fc / ((fa - fb) * (fa - fc))
                 + b * fa * fc / ((fb - fa) * (fb - fc))
                 + c * fa * fb / ((fc - fa) * (fc - fb)))
        elif fa != fb:
            s = b - fb * (b - a) / (fb - fa)
        else:
            s = (a + b) / 2.0

        cond1 = (s < (3 * a + b) / 4.0 or s > b)
        cond2 = (mflag and abs(s - b) >= abs(b - c) / 2.0)
        cond3 = ((not mflag) and abs(s - b) >= abs(c - d) / 2.0)
        cond4 = (mflag and abs(b - c) < tol)
        cond5 = ((not mflag) and abs(c - d) < tol)

        if cond1 or cond2 or cond3 or cond4 or cond5:
            s = (a + b) / 2.0
            mflag = True
        else:
            mflag = False

        fs = cdf_F_numba(s, q0, dt, v, Dq) - x_rand
        if fs == 0.0:
            return s

        d = c
        c = b
        fc = fb

        if fa * fs < 0.0:
            b = s
            fb = fs
        else:
            a = s
            fa = fs

        if abs(fa) < abs(fb):
            a, b = b, a
            fa, fb = fb, fa

        i += 1

    return b

In [22]:
@njit
def muestrear_q(q0, dt, v, Dq, x_rand):
    """
    Encuentra la nueva distancia q resolviendo F(q) - x_rand = 0.
    """

    q_max = math.sqrt(2.0 * Dq * dt) 
    if q_max < q0: 
        q_max = q0 * 1.5
        
    iteraciones = 0
    while (cdf_F_numba(q_max, q0, dt, v, Dq) - x_rand) <= 0.0:
        q_max *= 2.0
        iteraciones += 1
        if iteraciones > 100:
            q_max = 1e6
            break

    q_final = brent_numba(0.0, q_max, q0, dt, v, Dq, x_rand)
    
    return q_final

In [23]:
def pdf_teorica(q, q0, dt, v, Dq):
    
    raiz_com = np.sqrt(4*Dq*dt*np.pi)
    
    p1 = np.exp(-((q-q0-v*dt)**2)/ (4*Dq*dt) )* (1/raiz_com)
    
    p2 = np.exp(-((q+q0-v*dt)**2)/ (4*Dq*dt) )* (1/raiz_com) * np.exp(-(v*q0)/Dq)
    
    A = (v * q) / Dq
    B = (q + q0 + v * dt) / np.sqrt(4.0*Dq*dt)
    p3 = -(v / (2.0 * Dq)) * np.exp(A - B**2) * erfcx(B)
    
    return p1 + p2 + p3

In [24]:
@njit
def corregir_q_3d_pbc(x_new, y_new, z_new, x_old, y_old, z_old, phi, dt, v, Dq, bl, sigma):
    n_part = len(x_new)
    
    
    hubo_choque = True
    iteraciones_globales = 0
    max_iteraciones = 100 
    
    while hubo_choque and iteraciones_globales < max_iteraciones:
        hubo_choque = False 
        iteraciones_globales += 1
        
        for i in range(n_part - 1):
            for j in range(i + 1, n_part):
                
                xij = x_new[j] - x_new[i]
                yij = y_new[j] - y_new[i]
                zij = z_new[j] - z_new[i]
                
                xij -= bl*np.round(xij/bl)
                yij -= bl*np.round(yij/bl)
                zij -= bl*np.round(zij/bl)
                
                rij = math.sqrt(xij*xij + yij*yij + zij*zij)
                
                if rij < sigma:
                    hubo_choque = True 
                    
                    xij_old = x_old[j] - x_old[i]
                    yij_old = y_old[j] - y_old[i]
                    zij_old = z_old[j] - z_old[i]
                    
                    xij_old -= bl*np.round(xij_old/bl)
                    yij_old -= bl*np.round(yij_old/bl)
                    zij_old -= bl*np.round(zij_old/bl)
                    
                    rij_old = math.sqrt(xij_old*xij_old + yij_old*yij_old + zij_old*zij_old)
                    
                    q0 = max(rij_old - sigma, 0.0)
                    
                    e_x = xij_old / rij_old
                    e_y = yij_old / rij_old
                    e_z = zij_old / rij_old
                    
                    x_rand = np.random.uniform(0.0, 1.0)
                    if x_rand < 1e-15: x_rand = 1e-15
                    if x_rand > 1.0 - 1e-15: x_rand = 1.0 - 1e-15
                    
                    q_nuevo = muestrear_q(q0, dt, v, Dq, x_rand)
                    dq = q_nuevo - q0
                    
                    drx = (x_new[j] - x_old[j]) - (x_new[i] - x_old[i])
                    dry = (y_new[j] - y_old[j]) - (y_new[i] - y_old[i])
                    drz = (z_new[j] - z_old[j]) - (z_new[i] - z_old[i])
                    
                    dr_dot_e = drx*e_x + dry*e_y + drz*e_z
                    corr = dq - dr_dot_e
                    
                    x_new[i] -= 0.5 * corr * e_x
                    y_new[i] -= 0.5 * corr * e_y
                    z_new[i] -= 0.5 * corr * e_z

                    x_new[j] += 0.5 * corr * e_x
                    y_new[j] += 0.5 * corr * e_y
                    z_new[j] += 0.5 * corr * e_z
                    
    return iteraciones_globales

In [25]:
def init_config(x, y, z, n_part, phi, sigma=1.0):
    """
    Esta función coloca a las particulas en sus posiciones iniciales (Red Cúbica),
    ajustando dinámicamente el tamaño de la caja (bl) para evitar traslapes.
    """
    volumen_esfera = (np.pi / 6.0) * (sigma**3)
    bl = ((n_part * volumen_esfera) / phi)**(1/3)
    bl2 = bl / 2.0
    
    n_lado = math.ceil(n_part**(1/3))
    espaciado = bl / n_lado
    
    if espaciado < sigma:
        raise ValueError(f"ERROR: La fracción phi={phi} es muy alta. El espaciado ({espaciado:.3f}) es menor al diámetro ({sigma}).")
    
    x[0] = -bl2 + (espaciado / 2.0)
    y[0] = -bl2 + (espaciado / 2.0)
    z[0] = -bl2 + (espaciado / 2.0)
    
    for i in range(0, n_part - 1):
        x[i+1] = x[i] + espaciado
        y[i+1] = y[i]
        z[i+1] = z[i]
          
        if x[i+1] > bl2:
            x[i+1] = -bl2 + (espaciado / 2.0)
            y[i+1] = y[i+1] + espaciado
            z[i+1] = z[i]
            
            if y[i+1] > bl2:
                y[i+1] = -bl2 + (espaciado / 2.0)
                z[i+1] = z[i+1] + espaciado        

    return x, y, z, bl

In [26]:
@njit
def new_config(x, y, z, fx, fy, fz, n_part, dt):
    """
    Esta función actualiza las posiciones de lar partículas con las ec. de mov. de la dinámica Browniana

    """
    sigma = np.sqrt(2.0 * dt)
    
    for i in range(n_part):
        dx = sigma*np.random.normal(0.0,1.0)
        dy = sigma*np.random.normal(0.0,1.0)
        dz = sigma*np.random.normal(0.0,1.0)
        
        x[i] = x[i] + (fx[i]*dt) + dx
        y[i] = y[i] + (fy[i]*dt) + dy
        z[i] = z[i] + (fz[i]*dt) + dz

In [27]:
@njit
def calcular_histograma_gr(x, y, z, n_part, bl, dr, g, nr, rc):
    """
    Calcula el histograma de distancias para el g(r) en 3D.
    Usa un doble bucle asimétrico y la Convención de la Imagen Mínima.
    """
    for i in range(n_part - 1):
        for j in range(i + 1, n_part):
            
            xij = x[j] - x[i]
            yij = y[j] - y[i]
            zij = z[j] - z[i]
            
            xij -= bl * np.round(xij / bl)
            yij -= bl * np.round(yij / bl)
            zij -= bl * np.round(zij / bl)
            
            rij = math.sqrt(xij**2 + yij**2 + zij**2)
            
            if rij <= rc:

                m = int(rij / dr)
                if m < nr:
                    g[m] += 2.0 

In [28]:
def calculo_sq(x, y, z, q_vectores):
    """
    Esta función sirve para calcular el factor de estructura
    """
    
    N = len(x)
    posiciones = np.column_stack((x,y,z))
    fases = np.dot(posiciones, q_vectores.T)
    
    suma_cos = np.sum(np.cos(fases), axis = 0)
    suma_sin = np.sum(np.sin(fases), axis = 0)
    
    sq_por_direccion = (suma_cos**2 + suma_sin**2) / N
    
    return np.mean(sq_por_direccion)

In [29]:
def _msd_componente(cfg, chunk=64):
    """
    MSD de UNA componente (x, y o z) con el estimador de multiples origenes,
    calculado por FFT (algoritmo FCA de Kneller / Calandrini).
    """
    n, npart = cfg.shape #fotogramas y no. de particulas
    nfft = 2 * n #Define el tamaño del arreglo para la Transformada de Fourier
    idx = np.arange(n)

    ac = np.zeros(n)         # suma sobre particulas de la autocorrelacion
    Dsum = np.zeros(n + 1)   # suma sobre particulas de r[k]^2  (Dsum[n] = 0)

    for s in range(0, npart, chunk):
        b = cfg[:, s:s+chunk].astype(np.float64)
        b = b - b.mean(axis=0)
        F = np.fft.rfft(b, n=nfft, axis=0) #Hace una transformada rapida de Fourier
        ac += np.fft.irfft(F * F.conjugate(), n=nfft, axis=0)[:n].sum(axis=1)
        Dsum[:n] += np.einsum('ij,ij->i', b, b)

    Q = 2.0 * Dsum.sum()
    S1 = np.empty(n)
    for m in range(n):
        Q -= (Dsum[m-1] if m > 0 else 0.0) + Dsum[n-m]
        S1[m] = Q / (n - m)

    return (S1 - 2.0 * ac / (n - idx)) / npart


def calculo_D(cfgx, cfgy, cfgz, t):
    """
    Esta función sirve para obtener el coeficiente de difusión.
    """
    numero_de_fotos = cfgx.shape[0]

    tiempos_delta = t - t[0]

    msd = (_msd_componente(cfgx)
           + _msd_componente(cfgy)
           + _msd_componente(cfgz))
    msd[0] = 0.0

    D_t = np.zeros(numero_de_fotos)
    D_t[1:] = msd[1:] / (6.0 * tiempos_delta[1:])

    return tiempos_delta, msd, D_t

In [30]:
def escribir_frame(fh, x, y, z, bl, tiempo=0.0, sigma=1.0, tipo='C'):
    n = x.shape[0]; L = bl
    xw = (x + L/2.0) % L - L/2.0      # envolver a [-L/2, L/2)
    yw = (y + L/2.0) % L - L/2.0
    zw = (z + L/2.0) % L - L/2.0
    fh.write(f"{n}\n")
    fh.write(f'Lattice="{L:.6f} 0.0 0.0 0.0 {L:.6f} 0.0 0.0 0.0 {L:.6f}" '
             f'Origin="{-L/2:.6f} {-L/2:.6f} {-L/2:.6f}" '
             f'Properties=species:S:1:pos:R:3:radius:R:1 '
             f'pbc="T T T" Time={tiempo:.6f}\n')
    np.savetxt(fh, np.column_stack([xw, yw, zw, np.full(n, sigma/2.0)]),
               fmt=f'{tipo} %.5f %.5f %.5f %.4f')

In [31]:
def abrir_xyz(nombre):
    """Abre el archivo una sola vez; reabrirlo en cada frame es lentisimo."""
    return open(nombre, 'w')

In [32]:
resultados_g = {}
resultados_s = {}
resultados_d = {}
sigma = 1.0
phis = [0.1, 0.35]
carpeta = "resultados"
os.makedirs(carpeta, exist_ok=True)

for phi in phis:
    dt = dtt
    nprom = 0
    print('Iniciando termalización')
    x, y, z, bl = init_config(x, y, z, n_part, phi, sigma)
    dq = 2.0 * np.pi / bl
    nq = int(q_max/dq)
    q_valores = np.arange(1,nq+1)*dq
    rc = bl/2
    ngr = 0.0
    nsq = 0.0
    nr = int(rc/dr)  #no. cascaras, ahora que bl (y por tanto rc) ya se conoce
    fh_xyz = abrir_xyz(f"{carpeta}/traj_HS_phi{phi:.2f}_N{n_part}.xyz")

    r_dist = (np.arange(nr) * dr) + (dr / 2.0)
    dv = 4.0 * np.pi * (r_dist**2) * dr
    g = np.zeros(nr)
    s = np.zeros(nq)

    for i in tqdm(range(1, nct + 1), desc="Termalizando", unit="pasos"):

        x_old = np.copy(x)
        y_old = np.copy(y)
        z_old = np.copy(z)

        new_config(x, y, z, fx, fy, fz, n_part, dt)
        corregir_q_3d_pbc(x, y, z, x_old, y_old, z_old, phi, dt, v, Dq, bl, sigma)
        if i % ncxyz == 0:
            escribir_frame(fh_xyz, x, y, z, bl, tiempo=dt*i, sigma=sigma)

    print('Termalización completada, iniciando construcción de g(r), S(q) y MSD')

    # ---------------- Producción / toma de datos ----------------
    dt = dtp

    for i in tqdm(range(1, ncp + 1), desc="Recolectando datos", unit="pasos"):

        x_old = np.copy(x)
        y_old = np.copy(y)
        z_old = np.copy(z)

        new_config(x, y, z, fx, fy, fz, n_part, dt)
        corregir_q_3d_pbc(x, y, z, x_old, y_old, z_old, phi, dt, v, Dq, bl, sigma)
        
        if i % ncep == 0:

            t[nprom] = dt * i
            nprom += 1

            cfgx[nprom-1, :] = x.copy()
            cfgy[nprom-1, :] = y.copy()
            cfgz[nprom-1, :] = z.copy()
            
        if i % ncgr == 0:
                calcular_histograma_gr(x, y, z, n_part, bl, dr, g, nr, rc)
                ngr += 1     
        
        
        if i % ncsq == 0:
                
                for j, q_mag in enumerate(q_valores):
                        q_vectores_actual = q_mag * direcciones_base
                        s[j] += calculo_sq(x, y, z, q_vectores_actual)
                nsq +=1
            

        if i % ncxyz == 0:
            escribir_frame(fh_xyz, x, y, z, bl, tiempo=dt*i, sigma=sigma)

    print('Simulación completada')

    # ---------------- Normalización final ----------------
    rho = n_part / bl**3
    g_norm = g / (rho * dv * n_part * ngr)
    resultados_g[phi] = g_norm.copy()

    s_final = s / nsq
    resultados_s[phi] = s_final.copy()

    t_delta, msd_final, coef_difusion = calculo_D(cfgx, cfgy, cfgz, t)
    resultados_d[phi] = coef_difusion.copy()

    etiqueta = f"HS_phi{phi:.2f}_N{n_part}" 

    np.savetxt(f"{carpeta}/gr_{etiqueta}.csv",
            np.column_stack([r_dist, resultados_g[phi]]),
            header="r,g_r", delimiter=",", comments='')

    np.savetxt(f"{carpeta}/sq_{etiqueta}.csv",
            np.column_stack([q_valores, resultados_s[phi]]),
            header="q,S_q", delimiter=",", comments='')

    np.savetxt(f"{carpeta}/Dt_{etiqueta}.csv",
            np.column_stack([t_delta, coef_difusion, msd_final]),
            header="t,D_t,MSD", delimiter=",", comments='')
    
    fh_xyz.close()

Iniciando termalización


Termalizando: 100%|██████████| 100000/100000 [09:25<00:00, 176.81pasos/s]


Termalización completada, iniciando construcción de g(r), S(q) y MSD


Recolectando datos: 100%|██████████| 500000/500000 [48:11<00:00, 172.94pasos/s]


Simulación completada
Iniciando termalización


Termalizando: 100%|██████████| 100000/100000 [16:34<00:00, 100.53pasos/s]


Termalización completada, iniciando construcción de g(r), S(q) y MSD


Recolectando datos: 100%|██████████| 500000/500000 [1:38:02<00:00, 85.00pasos/s] 


Simulación completada
